# Canada EA Archive → your Google Drive

Downloads environmental-assessment documents from the catalogues in
[towertonhunt/ontario-rea-map](https://github.com/towertonhunt/ontario-rea-map)
straight into your Google Drive, using **your** storage.

**How to run:** Runtime → Run all. The first cell asks permission to access
your Drive (Google's own prompt). Safe to re-run any time — already-downloaded
files are skipped, so it resumes where it left off.

**Tiers** (set in the config cell):
- `1` — decision statements, certificates, approvals, conditions (~GBs, highest value)
- `2` — + main EA/EIS reports and appendices
- `3` — everything in the catalogues (largest)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#@title Config
TIER = 1            #@param [1, 2, 3] {type:"raw"}
JURISDICTIONS = ['federal', 'bc', 'ns', 'qc']   # which registries to archive
BRANCH = 'claude/mac-mini-connection-ceehl5'
DEST = '/content/drive/MyDrive/Canada-EA-Archive'
MAX_GB_THIS_SESSION = 40    # stop cleanly after this many GB (resume later)


In [ ]:
# Fetch the document catalogues from the repo (shallow, catalogues only)
import os, subprocess
if not os.path.isdir('/content/repo'):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
                    '--filter=blob:none', '--sparse',
                    'https://github.com/towertonhunt/ontario-rea-map', '/content/repo'],
                   check=True)
    subprocess.run(['git', '-C', '/content/repo', 'sparse-checkout', 'set',
                    'data/docs', 'data/projects_canada.geojson'], check=True)
print('catalogues ready')


In [ ]:
import json, re, glob, time, urllib.request, urllib.parse, csv, os

TIER1 = re.compile(r'decision statement|certificate|approval|conditions?|'
                   r'schedule [ab]|amendment', re.I)
TIER2 = re.compile(r'environmental (impact )?(statement|assessment)|EIS|'
                   r'main report|comprehensive study|final report|appendix', re.I)

def tier_of(title, category=''):
    s = f'{title} {category}'
    if TIER1.search(s): return 1
    if TIER2.search(s): return 2
    return 3

def safe(s, n=120):
    return re.sub(r'[^A-Za-z0-9._ -]+', '_', str(s))[:n].strip() or 'untitled'

manifest_path = f'{DEST}/manifest.csv'
os.makedirs(DEST, exist_ok=True)
done = set()
if os.path.exists(manifest_path):
    with open(manifest_path) as f:
        done = {row[0] for row in csv.reader(f) if row}
print(f'{len(done)} documents already archived')

UA = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'}
budget = MAX_GB_THIS_SESSION * 1024**3
spent = fetched = failed = skipped = 0

with open(manifest_path, 'a', newline='') as mf:
    writer = csv.writer(mf)
    for jur in JURISDICTIONS:
        for cat_file in sorted(glob.glob(f'/content/repo/data/docs/{jur}/*.json')):
            cat = json.load(open(cat_file))
            proj = safe(cat.get('project') or os.path.basename(cat_file)[:-5])
            for doc in cat['docs']:
                url = doc.get('url')
                if not url or url in done:
                    skipped += 1
                    continue
                if tier_of(doc.get('title',''), doc.get('category','')) > TIER:
                    continue
                folder = f'{DEST}/{jur}/{proj}'
                os.makedirs(folder, exist_ok=True)
                fname = safe(doc.get('title', 'doc'))
                dest = f'{folder}/{fname}'
                try:
                    req = urllib.request.Request(url, headers=UA)
                    with urllib.request.urlopen(req, timeout=120) as r:
                        ct = r.headers.get('Content-Type', '')
                        ext = ('.pdf' if 'pdf' in ct else
                               '.html' if 'html' in ct else '')
                        if not dest.endswith(('.pdf', '.html')):
                            dest += ext
                        data = r.read()
                    open(dest, 'wb').write(data)
                    spent += len(data)
                    fetched += 1
                    writer.writerow([url, dest, len(data)])
                    mf.flush()
                except Exception as e:
                    failed += 1
                    if failed < 20:
                        print('FAIL', url, str(e)[:80])
                if fetched % 50 == 0 and fetched:
                    print(f'{fetched} fetched, {spent/1e9:.2f} GB, {failed} failed')
                if spent > budget:
                    raise SystemExit(f'Session budget reached ({spent/1e9:.1f} GB) — re-run later to resume.')
                time.sleep(0.3)

print(f'DONE tier {TIER}: {fetched} fetched ({spent/1e9:.2f} GB), {failed} failed, {skipped} already archived')
